# Atención, paso a paso

**Explorador de Hespérides · Capítulo 5**

Ampliación programada sobre los conceptos de los notebooks D2L de este capítulo.

Los vectores son didácticos y están escritos en la celda: no son embeddings aprendidos del castellano. La fila seleccionada permite seguir puntuaciones, softmax, productos por valores y suma final. La máscara causal deja disponible la posición actual y bloquea las posteriores. Variar temperatura aquí modifica la atención, no el sampling de un modelo del lenguaje.

![Ilustración conceptual](../recursos/ilustraciones/capitulo_5.png)

*Ilustración conceptual generada con ImageGen. Los resultados cuantitativos son los del código.*

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# Vectores didácticos fijos: inspeccionamos el mecanismo, no un modelo del lenguaje entrenado.
tokens=['El','gato','mira','la','luna']
Q=torch.tensor([[1.,0.,0.],[0.,1.,0.],[.2,.8,.5],[1.,.1,0.],[0.,.3,1.]])
K=torch.tensor([[1.,0.,0.],[0.,1.,.2],[.2,.5,.5],[.8,.1,0.],[0.,.2,1.]])
V=torch.tensor([[1.,0.],[0.,1.],[.5,.5],[.8,.2],[.1,.9]])

def calcular_atencion(temperatura,causal):
    scores=Q@K.T/np.sqrt(Q.shape[1])/temperatura
    if causal:scores=scores.masked_fill(torch.triu(torch.ones(5,5,dtype=torch.bool),diagonal=1),-torch.inf)
    A=torch.softmax(scores,dim=-1)
    assert torch.allclose(A.sum(-1),torch.ones(5))
    if causal:assert torch.equal(torch.triu(A,diagonal=1),torch.zeros_like(A))
    return scores,A,A@V

def ver_atencion(consulta=2,temperatura=1.,causal=True):
    scores,A,O=calcular_atencion(temperatura,causal)
    fig,axes=plt.subplots(1,4,figsize=(14,3.4),gridspec_kw={'width_ratios':[1,1,1,.8]})
    axes[0].imshow(np.ma.masked_invalid(scores.numpy()),cmap='coolwarm',vmin=-2,vmax=2)
    axes[1].imshow(A,cmap='YlGnBu',vmin=0,vmax=1)
    for ax,titulo in zip(axes[:2],['QKᵀ / √d / temperatura','Softmax por fila']):
        ax.set(xticks=range(5),yticks=range(5),xticklabels=tokens,yticklabels=tokens,title=titulo,xlabel='Claves',ylabel='Consultas')
        ax.add_patch(plt.Rectangle((-.5,consulta-.5),5,1,fill=False,edgecolor='#D99B18',lw=2))
    contribuciones=A[consulta,:,None]*V
    axes[2].imshow(contribuciones,cmap='YlGnBu',vmin=0,vmax=1,aspect='auto')
    axes[2].set(yticks=range(5),yticklabels=tokens,xticks=[0,1],title='Peso × valor',xlabel='Componente de V')
    axes[3].bar([0,1],O[consulta],color=['#087E8B','#D99B18'])
    axes[3].set(ylim=(0,1),xticks=[0,1],title=f'Suma para «{tokens[consulta]}»',xlabel='Componente de salida')
    fig.tight_layout();plt.show()

interact(ver_atencion,consulta=widgets.IntSlider(value=2,min=0,max=4,description='Consulta',continuous_update=False),
         temperatura=widgets.FloatSlider(value=1,min=.1,max=3,step=.1,description='Temperatura',continuous_update=False),
         causal=True);


## Vista de referencia

Esta figura conserva el estado inicial también en una exportación sin kernel. Los controles anteriores se utilizan en Jupyter.

In [ ]:
ver_atencion(2,1.,True)

## Comprobación

Modifica un control cada vez y describe qué cambia y qué permanece constante. Compara tu observación con las preguntas del capítulo.